In [ ]:
[gtex_flip_simulation]
parameter: genofile = paths
# One .bim file per region; each region is processed independently.
parameter: cwd = path("output")
parameter: job_size = 30
parameter: walltime = "100h"
parameter: mem = "80G"
parameter: numThreads = 2

# ---- Simulation knobs --------------------------------------------------------
parameter: n_causal = 1        # number of causal variants per region
parameter: h2g = 0.05          # per-region heritability (raised vs UKB since GTEx n is much smaller)
parameter: total_h2g = True    # interpret h2g as total (vs per-causal)

# ---- Flip knobs --------------------------------------------------------------
parameter: miss_prop = 0.01    # fraction of non-causal variants to flip
parameter: miss_fac  = -1.0    # multiplier applied to flipped z / beta

parameter: container = ""

input: genofile, group_by = 1
output: f'{cwd:a}/miss_prop_{miss_prop}_miss_fac_{miss_fac}/sample_{_index}_ncausal_{n_causal}_h2g_{h2g}_gtex_flip.rds'
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime,
      mem = mem, cores = numThreads,
      tags = f'{step_name}_{_output[0]:bn}'

R:  expand = '${ }', stdout = f"{_output:n}.stdout", stderr = f"{_output:n}.stderr", container = container

    # -------------------------------------------------------------------------
    # Libraries and helpers
    # -------------------------------------------------------------------------
    library(plink2R)
    library(tidyverse)
    library(MASS)
    library(crayon)
    library(pecotmr)
    library(susieR)
    library(vroom)
    library(pgenlibr)
    
    source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/simulate_linreg.R")
    source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/misc.R")

    # pecotmr GitHub version (conda not yet updated)
    for (f in list.files("/sc/arion/projects/ad-omics/haochen/rss_qc/pecotmr/R/",
                         full.names = TRUE, pattern = "\\.R$")) source(f)
    select <- dplyr::select

    # -------------------------------------------------------------------------
    # Parse region id `i` from the GTEx .bim filename
    # -------------------------------------------------------------------------
    file <- ${_input:r}
    i <- file |>
        str_remove("\\.bim$") |>
        str_remove(".*/gtex_")

    # =========================================================================
    # 1. Load GTEx genotypes (the only panel used here)
    # =========================================================================
    gtex_path <- paste0("/sc/arion/projects/ad-omics/haochen/rss_qc/genotypes/gtex_trimed_random/gtex_", i)
    gtex_data <- plink_process(gtex_path)

    # =========================================================================
    # 2. Use all GTEx individuals and simulate the phenotype
    # =========================================================================
    # GTEx EUR n is ~600-800 — no sub-sampling, use the whole panel.
    Xmat <- gtex_data$bed

    # Guard: regions with too few SNPs are not informative for fine-mapping.
    if (ncol(Xmat) < 500) {
        stop("Too few SNPs — terminating job")
    }

    n_samples <- nrow(Xmat)
    p         <- ncol(Xmat)

    ntrait  <- 1
    ncausal <- ${n_causal}
    h2g     <- ${h2g}

    set.seed(123)

    # Sample causal SNPs and effect sizes; build the per-SNP effect vector beta.
    causal_index <- sample_causal_index(Xmat = Xmat, ncausal = ncausal)
    B_mat <- matrix(0, nrow = p, ncol = ntrait)
    beta  <- sim_beta_fix_variant(
        G            = Xmat,
        causal_index = causal_index,
        is_h2g_total = FALSE
    )
    B_mat[, 1] <- beta

    # Simulate the trait: y = X beta + e, with var(e) calibrated to hit h2g.
    phenotype <- sim_multi_traits(
        G            = Xmat,
        B            = B_mat,
        h2g          = h2g,
        is_h2g_total = FALSE
    )$P

    # =========================================================================
    # 3. Compute in-sample sumstats and LD
    # =========================================================================
    sumstat <- calculate_sumstat(Xmat, as.numeric(phenotype[, 1])) |>
        separate(
            SNP,
            into   = c("chrom", "pos", "A2", "A1"),
            sep    = ":",
            remove = FALSE
        ) |>
        mutate(
            chrom      = as.numeric(str_remove(chrom, "^chr")),
            pos        = as.integer(pos),
            original_z = z,                 # un-flipped copy for diagnostics
            original_b = Beta
        ) |>
        mutate(
            variants_id_original = paste0("chr", chrom, ":", pos, ":", A2, ":", A1),
            variant_id           = variants_id_original
        )

    LD_in_sample <- get_cormat(Xmat)

    # =========================================================================
    # 4. Deliberately flip a subset of non-causal variants
    # =========================================================================
    # Sample miss_prop * p indices uniformly, excluding the causal SNPs so that
    # the truth is preserved and only "noise" variants get sign-corrupted.
    miss_prop <- ${miss_prop}
    miss_fac  <- ${miss_fac}

    set.seed(123)
    candidate_idx <- setdiff(seq_len(p), causal_index)
    n_flip        <- round(miss_prop * p)
    flip_index    <- sample(candidate_idx, size = n_flip, replace = FALSE)

    sumstat_flipped <- sumstat
    sumstat_flipped$z[flip_index]    <- miss_fac * sumstat_flipped$z[flip_index]
    sumstat_flipped$Beta[flip_index] <- miss_fac * sumstat_flipped$Beta[flip_index]

    # =========================================================================
    # 5. Save bundled output for downstream SuSiE-RSS fitting
    # =========================================================================
    gtex_input <- list(
        sumstat          = sumstat_flipped,   # z / Beta with planted flips
        sumstat_original = sumstat,           # untouched reference
        LD               = LD_in_sample,
        flip_index       = flip_index
    )

    output_list <- list(
        gtex_input = gtex_input,
        true_vars  = causal_index,
        n_samples  = n_samples,
        n_snp      = p,
        miss_prop  = miss_prop,
        miss_fac   = miss_fac,
        input_path = gtex_path
    )

    data_out_path <- ${_output:r}
    dir.create(dirname(data_out_path), recursive = TRUE, showWarnings = FALSE)
    saveRDS(output_list, data_out_path)

In [ ]:
work_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/Allele_flip_Gtex_Simulation/"
mkdir -p ${work_dir}
mkdir -p ${work_dir}/code
mkdir -p ${work_dir}/log
cd ${work_dir}/code

# Create the base_script file and write the bash code into it
cat << 'EOF' > base_script
#!/bin/bash -l
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J allele_flip_sim
#BSUB -n 3
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=50000]
#BSUB -W 72:00
#BSUB -o /WORK_DIR/log/allele_flip_sim.%J.out
#BSUB -e /WORK_DIR/log/allele_flip_sim.%J.err
export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:$PATH:/usr/bin:/bin


sos run "/sc/arion/projects/ad-omics/haochen/rss_qc/2026_Simulation_Sinai/02.Fig1B.Allele_Flip_Simulation.ipynb" gtex_flip_simulation \
    --genofile $(echo /sc/arion/projects/ad-omics/haochen/rss_qc/genotypes/gtex_trimed_random/*.bim) \
    --n_causal CAUSAL --mem 70G --h2g H2G --miss_fac MISSFAC --miss_prop MISSPROP \
    --cwd WORK_DIR/result/h2g_H2G/causal_CAUSAL/
EOF


miss_fac=-1.0

base_sh="base_script"
for miss_prop in 0.001 0.01 0.1; do
    for causal in 1 2 3; do
       for h2g in 0.05 0.1 0.15; do
            output_script="miss_prop_${miss_prop}_miss_fac_${miss_fac}_causal_${causal}_h2g_${h2g}.sh"
            cat ${base_sh}| sed "s|MISSPROP|${miss_prop}|g" |sed "s|MISSFAC|${miss_fac}|g" |sed "s|CAUSAL|${causal}|g" |sed "s|WORK_DIR|${work_dir}|g" |sed "s|H2G|${h2g}|g"   > ${output_script}
            sb ${output_script}
        done
    done
done

In [ ]:
[krigging_fix]
parameter: genofile = paths
# pheno_file: give genotype file (in plink)，we can read the gentype matrix. These files are separated by TADs.
parameter: cwd = path("output")
parameter: job_size = 30
parameter: walltime = "100h"
parameter: mem = "40G"
parameter: numThreads = 1
# specify the number of causal variants
parameter: likelihood_thres = 10
parameter: LD_thres = 0.6
parameter: weight_thres = 0.6
# specify the number of traits (phenotypes)
parameter: container = ""
input: genofile, group_by = 1
output: f'{cwd:a}/{_input[0]:bn}_krigging_fix.rds'
R:  expand = '${ }', stdout = f"{_output:n}.stdout", stderr = f"{_output:n}.stderr", container = container 
    
    library(plink2R)
    library(tidyverse)
    library(MASS)
    library(crayon)
    library(pecotmr)
    library(susieR)
    library(vroom)
    library(pgenlibr)
    library(Rfast)

    # pecotmr GitHub version (conda not yet updated)
    for (f in list.files("/sc/arion/projects/ad-omics/haochen/rss_qc/pecotmr/R/",
                         full.names = TRUE, pattern = "\\.R$")) source(f)
    
    # pecotmr GitHub version (conda not yet updated)
    for (f in list.files("/sc/arion/projects/ad-omics/haochen/rss_qc/susieR/R/",
                         full.names = TRUE, pattern = "\\.R$")) source(f)
    select <- dplyr::select
    
    # Project utilities
    source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/simulate_linreg.R")
    source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/misc.R")
    source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/finemap_qc_ver2.r")
    path <- ${_input:r} 
    
    
    finemap_flip_susie_rss <- function(path) {
      dat       <- readRDS(path)
      n         <- dat$n_samples
      true_vars <- dat$true_vars
      LD        <- dat$gtex_input$LD
    
      z_flipped  <- dat$gtex_input$sumstat$z
      z_original <- dat$gtex_input$sumstat_original$z
    
      # ----- Kriging-based sign correction on the flipped z -----
      krig <- kriging_rss(z = z_flipped, R = LD, n = n)
      krig_outlier <- krig$conditional_dist %>%
        mutate(index = row_number()) %>%
        filter(abs(z) > 2 & logLR > 2)
    
      z_kriging <- z_flipped
      z_kriging[krig_outlier$index] <- -z_kriging[krig_outlier$index]
    
      # ----- susie_rss for each z version -----
      result_list <- list(
        flipped_res     = susie_rss(z = z_flipped,  R = LD, n = n),
        original_res    = susie_rss(z = z_original, R = LD, n = n),
        kriging_res     = susie_rss(z = z_kriging,  R = LD, n = n),
        kriging_outlier = krig_outlier
      )
    
      output <- list(
        result_list = result_list,
        n_sample    = n,
        input_path  = path,
        true_vars   = true_vars
      )
    
      return(output)
    }
    
    
    finemap_result = finemap_flip_susie_rss(path)
    
    saveRDS(finemap_result, ${_output:r})  

In [ ]:
work_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/Fine_Flip/"
mkdir -p ${work_dir}
mkdir -p ${work_dir}/code
mkdir -p ${work_dir}/log
cd ${work_dir}/code

# Create the base_script file and write the bash code into it
cat << 'EOF' > base_script
#!/bin/bash -l
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J Flip_fine
#BSUB -n 3
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=30000]
#BSUB -W 42:00
#BSUB -o /WORK_DIR/log/JOB_NAME.%J.out
#BSUB -e /WORK_DIR/log/JOB_NAME.%J.err

export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:$PATH:/usr/bin:/bin


sos run "/sc/arion/projects/ad-omics/haochen/rss_qc/2026_Simulation_Sinai/02.Fig1B.Allele_Flip_Simulation.ipynb"  krigging_fix \
    --genofile `ls /sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/Allele_flip_Gtex_Simulation/result/h2g_H2G/causal_CAUSAL/miss_prop_MISSPROP_miss_fac_MISSFAC/*.rds` \
    --mem 40G \
    --cwd WORK_DIR/result/h2g_H2G/causal_CAUSAL/miss_prop_MISSPROP_miss_fac_MISSFAC/
EOF


miss_fac=-1.0

base_sh="base_script"
for miss_prop in 0.001 0.01 0.1; do
    for causal in 1 2 3; do
       for h2g in 0.05 0.1 0.15; do
            output_script="miss_prop_${miss_prop}_miss_fac_${miss_fac}_causal_${causal}_h2g_${h2g}.sh"
            cat ${base_sh}| sed "s|MISSPROP|${miss_prop}|g" |sed "s|MISSFAC|${miss_fac}|g" |sed "s|CAUSAL|${causal}|g" |sed "s|WORK_DIR|${work_dir}|g" |sed "s|H2G|${h2g}|g"   > ${output_script}
            sb ${output_script}
        done
    done
done

In [ ]:
work_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/Fine_Flip/"
mkdir -p ${work_dir}/summary
cd ${work_dir}/summary

# Create the base_script file and write the bash code into it
cat << 'EOF' > base_script
#!/bin/bash -l
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J krig_summary
#BSUB -n 2
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=30000]
#BSUB -W 16:00
#BSUB -o WORK_DIR/summary/summary.%J.out
#BSUB -e WORK_DIR/summary/summary.%J.err


export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:$PATH:/usr/bin:/bin

sos run "/sc/arion/projects/ad-omics/haochen/rss_qc/2026_Simulation_Sinai/10.Fig1B_3_S2_S5_S6.Finemapping_Summary.ipynb" susie_bias_summary \
    --folder WORK_DIR/result/ \
    --cwd WORK_DIR/summary


EOF

base_sh="base_script"

output_script="summary_susie.sh"
cat ${base_sh} |sed "s|WORK_DIR|${work_dir}|g"    > ${output_script}
sb ${output_script}